# 02 — Feature Extraction: Time, Frequency & Time-Frequency Domains

Extracts the multi-domain feature set used to train the CNN-LSTM model.

| Domain | Features |
|---|---|
| Time (TD) | Mean, Pearson skewness, kurtosis per channel |
| Frequency (FD) | Band energy ratio, spectral kurtosis, spectral contrast |
| Combined T&FD | Best-performing config — ~97% accuracy |

## 1. Time-Domain Features
Mean, Pearson skewness, and kurtosis extracted per DAS channel.

In [ ]:
import pandas as pd
import numpy as np

def extract_time_domain_features(df):
    """Compute time-domain statistical features per DAS channel."""
    rows = []
    for col in df.columns:
        x = df[col].values
        mean_val   = np.mean(x)
        std_val    = np.std(x) + 1e-10
        median_val = np.median(x)
        # Pearson skewness — used in paper (robust to outliers)
        pearson_skew = (mean_val - median_val) / std_val
        # Kurtosis (4th standardised moment)
        kurt = np.mean((x - mean_val) ** 4) / (std_val ** 2) ** 2
        rows.append([mean_val, pearson_skew, kurt])
    return pd.DataFrame(rows, columns=["mean", "skewness", "kurtosis"])

print("Time-domain extractor ready — features: mean, Pearson skewness, kurtosis")

## 2. Frequency-Domain Features
Spectral features computed via FFT on each channel.

In [ ]:
def extract_frequency_domain_features(df):
    """Compute frequency-domain spectral features per DAS channel."""
    rows = []
    for col in df.columns:
        x = df[col].values
        mag   = np.abs(np.fft.fft(x))[:len(x)//2]
        freqs = np.fft.fftfreq(len(x))[:len(x)//2]
        total = np.sum(mag) + 1e-10

        centroid = np.sum(freqs * mag) / total
        mid_idx  = len(mag) // 4
        ber      = np.sum(mag[:mid_idx]**2) / (np.sum(mag[mid_idx:]**2) + 1e-10)
        mu, sigma = np.mean(mag), np.std(mag) + 1e-10
        spec_kurt = np.mean((mag - mu)**4) / sigma**4
        contrast  = float(np.max(mag) - np.min(mag))

        rows.append([ber, spec_kurt, contrast])
    return pd.DataFrame(rows, columns=["band_energy_ratio","spectral_kurtosis","spectral_contrast"])

print("Frequency-domain extractor ready — features: BER, spectral kurtosis, contrast")

## 3. Build Combined T&FD Feature Matrix

In [ ]:
def build_feature_matrix(df, label_col=None):
    """Full pipeline: extract T&FD features and combine into training matrix."""
    print("Extracting time-domain features...")
    td = extract_time_domain_features(df)
    print("Extracting frequency-domain features...")
    fd = extract_frequency_domain_features(df)
    combined = pd.concat([td, fd], axis=1)
    if label_col and label_col in df.columns:
        combined[label_col] = df[label_col].values
    print(f"Feature matrix: {combined.shape} | columns: {list(combined.columns)}")
    return combined

# Example:
# df = pd.read_csv("data/processed/raw_das_signal.csv")
# features = build_feature_matrix(df, label_col="condition")
# features.to_csv("data/processed/Training_Data.csv", index=False)
print("T&FD pipeline ready.")

## 4. Feature Distribution Visualisation

In [ ]:
import matplotlib.pyplot as plt

def plot_feature_distributions(feature_df, label_col="condition"):
    features = [c for c in feature_df.columns if c != label_col]
    fig, axes = plt.subplots(1, len(features), figsize=(4*len(features), 4))
    for ax, feat in zip(axes, features):
        if label_col in feature_df.columns:
            for lbl in feature_df[label_col].unique():
                ax.hist(feature_df[feature_df[label_col]==lbl][feat],
                        bins=30, alpha=0.5, label=str(lbl))
            ax.legend(fontsize=8)
        else:
            ax.hist(feature_df[feat], bins=30)
        ax.set_title(feat, fontsize=10)
        ax.set_xlabel("Value")
    plt.suptitle("Feature Distributions by Condition Class", fontweight="bold")
    plt.tight_layout()
    plt.savefig("results/figures/feature_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Visualisation function ready.")